In [1]:
import numpy as np
import pickle
from d3rlpy.dataset.components import Episode

dataset = "hopper-medium-expert-v2"
if "halfcheetah" in dataset:
    env_name = "HalfCheetah-v4"
elif "hopper" in dataset:
    env_name = "Hopper-v4"
elif "walker" in dataset:
    env_name = "Walker2d-v4"

#env_name = dataset.split("-")[0][0].upper() + dataset.split("-")[0][1:] + "-v5"
print(env_name)

pkl_path = f"../../../../master_thesis/reproducing_decision_transformer/gymnasium/data/{dataset}.pkl"

def convert_raw_episode(raw_ep):
    # Convert raw observations to a NumPy array and then to a list of individual observations.
    observations = np.array(raw_ep["observations"])

    # Ensure actions and rewards are NumPy arrays.
    actions = np.array(raw_ep["actions"])
    rewards = np.array(raw_ep["rewards"])
    # For rewards, ensure they have an extra dimension (T, 1)
    if rewards.ndim == 1:
        rewards = rewards.reshape(-1, 1)
    
    # Use the last element of "terminals" as the terminated flag.
    terminals = raw_ep["terminals"]
    if isinstance(terminals, (list, np.ndarray)):
        terminated = bool(terminals[-1])
    else:
        terminated = bool(terminals)
    
    return Episode(
        observations=observations,
        actions=actions,
        rewards=rewards,
        terminated=terminated
    )

def load_and_convert_episodes(pkl_path):
    with open(pkl_path, "rb") as f:
        raw_episodes = pickle.load(f)
    return [convert_raw_episode(ep) for ep in raw_episodes]

# Example usage:
episodes = load_and_convert_episodes(pkl_path=pkl_path)
print(f"Loaded {len(episodes)} episodes.")

from d3rlpy.dataset import ReplayBuffer, FIFOBuffer

buffer_impl = FIFOBuffer(limit=10000000)
replay_buffer = ReplayBuffer(buffer=buffer_impl, episodes=episodes)


/gpfs/data/fs72297/jklotz/.conda/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hopper-v4
Loaded 3213 episodes.
2025-07-10 20:30.47 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)])
2025-07-10 20:30.47 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.CONTINUOUS: 1>
2025-07-10 20:30.47 [info     ] Action size has been automatically determined. action_size=3


In [2]:
import gymnasium as gym
import d3rlpy
import argparse
# parser.add_argument("--dataset", type=str, default="hopper-medium-v0")
# parser.add_argument("--seed", type=int, default=1)
# parser.add_argument("--gpu", type=int)
# parser.add_argument("--compile", action="store_true")
# args = parser.parse_args()

args = argparse.Namespace()
args.dataset = dataset
args.seed = 1
args.gpu = 1
args.compile = False

import gym
env = gym.make(env_name)

#dataset, env = d3rlpy.datasets.get_dataset(args.dataset)

# fix seed
d3rlpy.seed(args.seed)
d3rlpy.envs.seed_env(env, args.seed)

if "halfcheetah" in args.dataset:
    target_return = 6000
elif "hopper" in args.dataset:
    target_return = 3800
elif "walker" in args.dataset:
    target_return = 5000
else:
    raise ValueError("unsupported dataset")

In [3]:
print(env.observation_space)

Box(-inf, inf, (11,), float64)


In [4]:
import argparse
from datetime import datetime
from typing import Optional, Union

import gym
import numpy as np
from torch.optim.lr_scheduler import CosineAnnealingLR

import d3rlpy
from d3rlpy.algos import CQL, IQL
from d3rlpy.dataset import InfiniteBuffer, ReplayBuffer
from d3rlpy.types import NDArray

from d3rlpy.logging import UnifiedFileAdapterFactory
import os

In [5]:
def relabel_dataset_rtg(
    buffer: InfiniteBuffer,
    q_algo: Union[CQL, IQL],
    k: int,
    num_action_samples: int,
) -> None:
    """
    Relabel RTG (reward-to-go) to the given dataset using the given Q-function.

    Args:
        buffer (InfiniteBuffer): Buffer holding trajectory dataset.
        q_algo (Union[CQL, IQL]): Trained Q-learning algoirthm.
        k (int): Context length for DT.
        num_action_samples (int): The number of action samples for
            V function estimation. Defaults to 10.
    """
    prev_idx = -1
    for n in range(buffer.transition_count):
        episode, idx = buffer._transitions[-n - 1]  # get transitions backwards
        if idx > prev_idx:
            # get values for all observations in the episode
            values = []
            for _ in range(num_action_samples):
                sampled_actions = q_algo.sample_action(episode.observations)
                values.append(
                    q_algo.predict_value(episode.observations, sampled_actions)
                )
            value = np.array(values).mean(axis=0)
            rewards = np.squeeze(episode.rewards, axis=1)
            rtg = 0
        else:
            start = max(0, idx - k + 1)
            rtg = rewards[idx] + np.maximum(rtg, value[idx + 1])  # relabel rtg
            relabelled_rewards = np.zeros_like(rewards)
            relabelled_rewards[idx] = rtg
            relabelled_rewards[start:idx] = rewards[start:idx]
            relabelled_episode = d3rlpy.dataset.components.Episode(
                observations=episode.observations,
                actions=episode.actions,
                rewards=np.expand_dims(relabelled_rewards, axis=1),
                terminated=episode.terminated,
            )
            buffer._transitions[-n - 1] = (relabelled_episode, idx)

        prev_idx = idx

    return


""" --------------------------------------------------------------------
    Fit offline RL algorithms to the given dataset.
-------------------------------------------------------------------- """


def fit_cql(
    dataset: ReplayBuffer,
    env: gym.Env[NDArray, int],
    gpu: Optional[int],
    log_postfix: str,
) -> CQL:
    """
    Fit the CQL algorithm to the given dataset and environment.

    Args:
        dataset (ReplayBuffer): Dataset for the training.
        env (gym.Env): The environment instance.
        gpu (Optional[int]): The GPU device ID..
        log_postfix (str): The postfix of experiment name.

    Return:
        Trained CQL agent.
    """
    encoder = d3rlpy.models.encoders.VectorEncoderFactory([256, 256, 256])

    if "medium-v0" in env.spec.id:
        conservative_weight = 10.0
    else:
        conservative_weight = 5.0

    cql = d3rlpy.algos.CQLConfig(
        actor_learning_rate=1e-4,
        critic_learning_rate=3e-4,
        temp_learning_rate=1e-4,
        actor_encoder_factory=encoder,
        critic_encoder_factory=encoder,
        batch_size=256,
        n_action_samples=10,
        alpha_learning_rate=0.0,
        conservative_weight=conservative_weight,
    ).create(device="cuda")

    cql.fit(
        dataset,
        n_steps=500000,#500000
        n_steps_per_epoch=1000,#1000
        save_interval=100,#50
        evaluators=None,#{"environment": d3rlpy.metrics.EnvironmentEvaluator(env)},
        experiment_name=f"CQL_{log_postfix}",
        with_timestamp=False,
        logger_adapter=UnifiedFileAdapterFactory(),

    )

    return cql


In [6]:
def fit_iql(
    dataset: ReplayBuffer,
    env: gym.Env[NDArray, int],
    gpu: Optional[int],
    log_postfix: str,
) -> IQL:
    """
    Fit the IQL algorithm to the given dataset and environment.

    Args:
        dataset (ReplayBuffer): Dataset for the training.
        env (gym.Env): The environment instance.
        seed (int): The random seed.
        gpu (Optional[int]): The GPU device ID.
        log_postfix (str): The postfix of experiment name.

    Return:
        Trained IQL agent.
    """
    reward_scaler = d3rlpy.preprocessing.ReturnBasedRewardScaler(
        multiplier=1000.0
    )

    iql = d3rlpy.algos.IQLConfig(
        actor_learning_rate=3e-4,
        critic_learning_rate=3e-4,
        batch_size=256,
        gamma=0.99,
        weight_temp=3.0,
        max_weight=100.0,
        expectile=0.7,
        reward_scaler=reward_scaler,
    ).create(device="cuda")

    # workaround for learning scheduler
    iql.build_with_dataset(dataset)
    assert iql.impl
    scheduler = CosineAnnealingLR(
        iql.impl._modules.actor_optim,  # pylint: disable=protected-access
        500000,
    )

    def callback(algo: d3rlpy.algos.IQL, epoch: int, total_step: int) -> None:
        scheduler.step()

    iql.fit(
        dataset,
        n_steps=500,#500000
        n_steps_per_epoch=10,#1000
        save_interval=5,#10
        callback=callback,
        evaluators={
            "environment": d3rlpy.metrics.EnvironmentEvaluator(env, n_trials=10)
        },
        experiment_name=f"IQL_{log_postfix}",
        with_timestamp=False,
        logger_adapter=UnifiedFileAdapterFactory(),

    )

    return iql

In [7]:
def fit_dt(
    dataset: ReplayBuffer,
    env: gym.Env[NDArray, int],
    target_return: int,
    context_size: int,
    gpu: Optional[int],
    log_postfix: str,
) -> None:
    """
    Fit decisiton transformer to the given dataset and environment.

    Args:
        dataset (MDPdataset): Dataset for the training.
        env (gym.Env): The environment instance.
        context_size (int): The context size of DT.
        gpu (Optional[int]): The GPU device ID.
        log_postfix (str): The postfix of experiment name.
    """

    # dt = d3rlpy.algos.DecisionTransformerConfig(
    #     batch_size=64,
    #     learning_rate=1e-4,
    #     optim_factory=d3rlpy.optimizers.AdamWFactory(
    #         weight_decay=1e-4,
    #         clip_grad_norm=0.25,
    #         lr_scheduler_factory=d3rlpy.optimizers.WarmupSchedulerFactory(
    #             warmup_steps=10000
    #         ),
    #     ),
    #     encoder_factory=d3rlpy.models.VectorEncoderFactory(
    #         [128],
    #         exclude_last_activation=True,
    #     ),
    #     observation_scaler=d3rlpy.preprocessing.StandardObservationScaler(),
    #     reward_scaler=d3rlpy.preprocessing.MultiplyRewardScaler(0.001),
    #     position_encoding_type=d3rlpy.PositionEncodingType.SIMPLE,
    #     context_size=context_size,
    #     num_heads=1,
    #     num_layers=3,
    #     max_timestep=1000,
    # ).create(device=gpu)

    # dt.fit(
    #     dataset,
    #     n_steps=1000,#100000
    #     n_steps_per_epoch=10,#1000
    #     save_interval=5,#10
    #     eval_env=env,
    #     eval_target_return=target_return,
    #     experiment_name=f"QDT_{log_postfix}",
    #     with_timestamp=False,
    # )

    
    dt = d3rlpy.algos.DecisionTransformerConfig(
        batch_size=64,
        learning_rate=1e-4,
        optim_factory=d3rlpy.optimizers.AdamWFactory(
            weight_decay=1e-4,
            clip_grad_norm=0.25,
            lr_scheduler_factory=d3rlpy.optimizers.WarmupSchedulerFactory(
                warmup_steps=10000#10000
            ),
        ),
        encoder_factory=d3rlpy.models.VectorEncoderFactory(
            [128],
            exclude_last_activation=True,
        ),
        observation_scaler=d3rlpy.preprocessing.StandardObservationScaler(),
        reward_scaler=d3rlpy.preprocessing.MultiplyRewardScaler(0.001),
        position_encoding_type=d3rlpy.PositionEncodingType.SIMPLE,
        context_size=20,
        num_heads=1,
        num_layers=3,
        max_timestep=1000,
        compile_graph=False,
    ).create(device="cuda")
    
    dt.fit(
        replay_buffer,
        n_steps=100000,#100000,
        n_steps_per_epoch=1000,#1000,
        save_interval=1,
        eval_env=env,
        eval_target_return=target_return,
        experiment_name=f"QDT_{args.dataset}_{args.seed}",
        n_trials=100,
        eval_gaps=1,
        logger_adapter=UnifiedFileAdapterFactory(),
        patience=15,
    )

In [ ]:
def main() -> None:
    from types import SimpleNamespace

    args = SimpleNamespace(
        dataset="hopper-medium-expert-v2",
        context_size=20,
        #model_file="/gpfs/data/fs72297/jklotz/programming/cloned_repos/forked_repos_for_master_thesis/d3rlpy/experiments/exp02_qdt/d3rlpy_logs/CQL_Hopper-v4_1_20250709212343/model_4950.d3",
        model_file=None,
        q_learning_type="cql",
        seed=1,
        num_action_samples=10,
        gpu=0,
        compile = False
    )
    # parser = argparse.ArgumentParser()
    # parser.add_argument("--dataset", type=str, default="hopper-medium-v2")
    # parser.add_argument("--context_size", type=int, default=20)
    # parser.add_argument("--model_file", type=str, default=None)
    # parser.add_argument(
    #     "--q_learning_type",
    #     type=str,
    #     default="cql",
    #     choices=["cql", "iql"],
    # )
    # parser.add_argument("--seed", type=int, default=1)
    # parser.add_argument("--num_action_samples", type=int, default=10)
    # parser.add_argument("--gpu", type=int)
    # args = parser.parse_args()

    if "halfcheetah" in args.dataset:
        env_name = "HalfCheetah-v4"
        target_return = 6000
    elif "hopper" in args.dataset:
        env_name = "Hopper-v4"
        target_return = 3600
    elif "walker" in args.dataset:
        env_name = "Walker2d-v4"
        target_return = 5000
        
    import gym
    env = gym.make(env_name)
    pkl_path = f"../../../../master_thesis/reproducing_decision_transformer/gymnasium/data/{args.dataset}.pkl"

    def convert_raw_episode(raw_ep):
        # Convert raw observations to a NumPy array and then to a list of individual observations.
        observations = np.array(raw_ep["observations"])
    
        # Ensure actions and rewards are NumPy arrays.
        actions = np.array(raw_ep["actions"])
        rewards = np.array(raw_ep["rewards"])
        # For rewards, ensure they have an extra dimension (T, 1)
        if rewards.ndim == 1:
            rewards = rewards.reshape(-1, 1)
        
        # Use the last element of "terminals" as the terminated flag.
        terminals = raw_ep["terminals"]
        if isinstance(terminals, (list, np.ndarray)):
            terminated = bool(terminals[-1])
        else:
            terminated = bool(terminals)
        
        return Episode(
            observations=observations,
            actions=actions,
            rewards=rewards,
            terminated=terminated
        )
    
    def load_and_convert_episodes(pkl_path):
        with open(pkl_path, "rb") as f:
            raw_episodes = pickle.load(f)
        return [convert_raw_episode(ep) for ep in raw_episodes]
    
    # Example usage:
    episodes = load_and_convert_episodes(pkl_path=pkl_path)
    print(f"Loaded {len(episodes)} episodes.")
    
    from d3rlpy.dataset import InfiniteBuffer, ReplayBuffer, FIFOBuffer
    

    buffer_impl = FIFOBuffer(100000)
    dataset = ReplayBuffer(buffer=buffer_impl, episodes=episodes)
    
    #dataset, env = d3rlpy.datasets.get_dataset(args.dataset)

    # create postfix of log directories
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    log_postfix = f"{env.spec.id}_{args.seed}_{timestamp}"

    # fix seed
    d3rlpy.seed(args.seed)
    d3rlpy.envs.seed_env(env, args.seed)

    # first fit Q-learning algorithm to the dataset
    if args.model_file is not None:
        # load model and assert type
        q_algo_loaded = d3rlpy.load_learnable(args.model_file)
        if not isinstance(q_algo_loaded, (CQL, IQL)):
            raise ValueError(
                "The loaded model is not an instance of CQL or IQL."
            )
        # cast to the expected type
        q_algo = q_algo_loaded
    else:
        if args.q_learning_type == "cql":
            q_algo = fit_cql(
                dataset=dataset,
                env=env,
                gpu=args.gpu,
                log_postfix=log_postfix,
            )
        elif args.q_learning_type == "iql":
            q_algo = fit_iql(
                dataset=dataset,
                env=env,
                gpu=args.gpu,
                log_postfix=log_postfix,
            )
        else:
            raise ValueError(f"invalid q_learning_type: {args.q_learning_type}")

    # relabel dataset RTGs with the learned value functions
    print("Relabeling dataset with RTGs...")
    inf_buffer = InfiniteBuffer()
    for episode in episodes:
        inf_buffer.append(episode,0)
    assert isinstance(inf_buffer, InfiniteBuffer)
    relabel_dataset_rtg(
        buffer=inf_buffer,
        q_algo=q_algo,
        k=args.context_size,
        num_action_samples=args.num_action_samples,
    )
    
    episodes = [episode for (episode, _) in inf_buffer._transitions]
    buffer_impl = FIFOBuffer(limit=10000000)
    replay_buffer = ReplayBuffer(buffer=buffer_impl, episodes=episodes)
    print(type(replay_buffer))
    # fit decision transformer to the relabeled dataset
    fit_dt(
        dataset=replay_buffer,
        env=env,
        target_return=target_return,
        context_size=args.context_size,
        gpu=args.gpu,
        log_postfix=log_postfix,
    )
main()


Loaded 3213 episodes.
2025-07-10 20:30.51 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)])
2025-07-10 20:30.51 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.CONTINUOUS: 1>
2025-07-10 20:30.51 [info     ] Action size has been automatically determined. action_size=3
2025-07-10 20:30.53 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]), action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]), reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]), action_space=<ActionSpace.CONTINUOUS: 1>, action_size=3)
2025-07-10 20:30.53 [debug    ] Building models...            
2025-07-10 20:30.53 [debug    ] Models have been built.     

Epoch 1/500: 100%|██████████| 1000/1000 [00:24<00:00, 41.34it/s, critic_loss=-62.1, conservative_loss=-64.4, alpha=1, actor_loss=-10.1, temp=0.957, temp_loss=3.57]

2025-07-10 20:31.17 [info     ] CQL_Hopper-v4_1_20250710203053: epoch=1 step=1000 epoch=1 metrics={'time_sample_batch': 0.0043609018325805665, 'time_algorithm_update': 0.019542587518692017, 'critic_loss': -62.183445808410646, 'conservative_loss': -64.47241333007813, 'alpha': 1.0, 'actor_loss': -10.158183187007904, 'temp': 0.9570802025794983, 'temp_loss': 3.562954094171524, 'time_step': 0.0240251145362854} step=1000



Epoch 2/500: 100%|██████████| 1000/1000 [00:21<00:00, 46.68it/s, critic_loss=-69.1, conservative_loss=-72.5, alpha=1, actor_loss=-25.7, temp=0.882, temp_loss=2.5]

2025-07-10 20:31.39 [info     ] CQL_Hopper-v4_1_20250710203053: epoch=2 step=2000 epoch=2 metrics={'time_sample_batch': 0.003660973072052002, 'time_algorithm_update': 0.017495909690856935, 'critic_loss': -69.05157055282592, 'conservative_loss': -72.52464971923828, 'alpha': 1.0, 'actor_loss': -25.800758178710936, 'temp': 0.8820398149490356, 'temp_loss': 2.497598034262657, 'time_step': 0.021270270347595214} step=2000


...


Epoch 296/500: 100%|██████████| 1000/1000 [00:32<00:00, 31.21it/s, critic_loss=-84.4, conservative_loss=-88.4, alpha=1, actor_loss=-418, temp=0.326, temp_loss=0.00332]

2025-07-10 23:01.37 [info     ] CQL_Hopper-v4_1_20250710203053: epoch=296 step=296000 epoch=296 metrics={'time_sample_batch': 0.005744814157485962, 'time_algorithm_update': 0.025890120267868043, 'critic_loss': -84.3942483291626, 'conservative_loss': -88.42750608062744, 'alpha': 1.0, 'actor_loss': -418.5033932800293, 'temp': 0.32622475731372835, 'temp_loss': 0.003279776024632156, 'time_step': 0.031780970335006715} step=296000



Epoch 297/500: 100%|██████████| 1000/1000 [00:33<00:00, 30.01it/s, critic_loss=-84.3, conservative_loss=-88.5, alpha=1, actor_loss=-418, temp=0.325, temp_loss=-0.000194]

2025-07-10 23:02.11 [info     ] CQL_Hopper-v4_1_20250710203053: epoch=297 step=297000 epoch=297 metrics={'time_sample_batch': 0.0059068574905395506, 'time_algorithm_update': 0.02700142216682434, 'critic_loss': -84.30480628204346, 'conservative_loss': -88.53751103210449, 'alpha': 1.0, 'actor_loss': -418.3622172241211, 'temp': 0.3251087028682232, 'temp_loss': -0.00029801466525532303, 'time_step': 0.03305334901809692} step=297000



Epoch 298/500:  97%|█████████▋| 969/1000 [00:22<00:00, 44.37it/s, critic_loss=-84.6, conservative_loss=-88.6, alpha=1, actor_loss=-419, temp=0.326, temp_loss=-0.00197] 